# ISLES'24 NCCT OOF evaluation — repaired and resumable

**Use Runtime → Change runtime type → CPU, then Runtime → Run all.**
This notebook uses the *existing*, completely trained 149-case Dataset504 NCCT nnU-Net results. It does not train a model.

**Why this version:** Earlier Notebook 12 failed before evaluation because `pip install -e .[evaluation]` encountered a metadata-generation failure. This notebook **does not install the project or its broad optional extras**: it imports project source directly and installs a narrow, wheel-only set of metric dependencies.

**Visible and persistent progress:** after each case the console prints `[completed/149] percent, fold, case and Dice`; the private Drive output folder stores `ncct_oof_progress.json`, `case_checkpoints/`, and `ncct_oof_resumable.log`. Rerun **Run all** after a disconnect: validated completed cases are reused. No patient-level outputs are committed to GitHub.

**Metrics:** the public ISLES'24 four-metric implementation without unavailable organizer-specific NCCT brain masking. Do not report these local out-of-fold results as identical to the hidden-test challenge ranking.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


## 1. Set up persistent logging and download the latest verified source


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, os, shutil, subprocess, sys

ROOT = Path("/content/drive/MyDrive/HybridStrokeSeg/ISLES2024")
PREPROCESSED = ROOT / "nnUNet_preprocessed"
RESULTS = ROOT / "nnUNet_results"
OUTPUT = ROOT / "results" / "ncct_oof_evaluation"
LOG = OUTPUT / "ncct_oof_resumable.log"
PROGRESS = OUTPUT / "ncct_oof_progress.json"
REPOZIP = Path("/content/HybridStrokeSeg-main.zip")
REPO = Path("/content/HybridStrokeSeg-Reproduction-main")

OUTPUT.mkdir(parents=True, exist_ok=True)
for path in (PREPROCESSED, RESULTS):
    if not path.is_dir():
        raise FileNotFoundError(f"Missing prepared data: {path}")

ENV = os.environ.copy()
ENV["PYTHONPATH"] = str(REPO / "src") + os.pathsep + ENV.get("PYTHONPATH", "")
ENV["PYTHONUNBUFFERED"] = "1"

def run_logged(step, command, *, cwd=None):
    command = [str(item) for item in command]
    print(f"\n=== {step} ===", flush=True)
    print("$", " ".join(command), flush=True)
    with LOG.open("a", encoding="utf-8") as log:
        log.write(f"\n[{datetime.now(timezone.utc).isoformat()}] {step}\n")
        log.write("$ " + " ".join(command) + "\n")
        log.flush()
        with subprocess.Popen(
            command, cwd=cwd, env=ENV, stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, text=True, bufsize=1,
        ) as process:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
                log.flush()
            code = process.wait()
        log.write(f"[{step}] exit_code={code}\n")
        log.flush()
        if code != 0:
            raise RuntimeError(
                f"{step} failed with exit code {code}. "
                f"Full failure log: {LOG}"
            )

print("Python:", sys.version)
print("Persistent log:", LOG)
print("Persistent progress:", PROGRESS)

if REPO.exists():
    shutil.rmtree(REPO)
if REPOZIP.exists():
    REPOZIP.unlink()
run_logged("Download repository", [
    "curl", "-fL", "--retry", "10", "--retry-delay", "3",
    "https://github.com/NTinkicht/HybridStrokeSeg-Reproduction/archive/refs/heads/main.zip",
    "-o", REPOZIP,
])
run_logged("Extract repository", ["unzip", "-q", REPOZIP, "-d", "/content"])
assert (REPO / "scripts" / "evaluate_isles2024_nnunet_oof.py").exists()


## 2. Install **only evaluation dependencies** from prebuilt wheels

The full repository and nnU-Net training dependencies are intentionally **not installed**. This avoids the source-package metadata generation that broke Notebook 12. Exact Panoptica version is fixed for reproducibility. If a wheel is unavailable, setup fails *with detailed progress logs*, rather than attempting an unverified source build.


In [ ]:
run_logged("Install NIfTI and component libraries (wheel-only)", [
    sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
    "--only-binary=:all:", "nibabel>=5.2,<6", "connected-components-3d>=3.12,<4",
    "surface-distance>=0.1,<1", "ruamel.yaml>=0.18,<1", "scikit-image>=0.23,<1", "SimpleITK>=2.3,<3", "rich>=13,<15",
])
run_logged("Install pinned Panoptica wheel without pulling broad optional builds", [
    sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
    "--only-binary=:all:", "--no-deps", "panoptica==0.5.22",
])


## 3. Small API smoke test before processing any clinical cases

This verifies that Panoptica imports successfully and returns an exact Dice of 1.0 for an identical synthetic pair. Any dependency problem is captured in the persistent log here rather than after loading 149 volumes.


In [ ]:
smoke_code = r"""
import numpy as np
from importlib.metadata import version
from hybridstrokeseg.isles2024_eval import evaluate_case
mask = np.zeros((7, 7, 7), dtype=np.uint8)
mask[2:5, 2:5, 2:5] = 1
value = evaluate_case(case_id="synthetic_smoke", fold=0,
                      reference=mask, prediction=mask.copy(),
                      voxel_volume_ml=0.001)
assert abs(value.dice - 1.0) < 1e-8, value
assert abs(value.lesion_f1 - 1.0) < 1e-8, value
print("Metric API PASS; Dice=1, lesion-F1=1")
print("Panoptica:", version("panoptica"))
"""
run_logged("Metric API smoke test", [sys.executable, "-c", smoke_code])


## 4. Evaluate or resume all 149 patients

The evaluator prints one persistent, timestamped progress line per case. The case checkpoints are validated against input file paths, sizes and modification times before being reused. Intermediate progress is saved every case, with failures recorded immediately.


In [ ]:
command = [
    sys.executable, "-u", "scripts/evaluate_isles2024_nnunet_oof.py",
    "--dataset-id", "504", "--configuration", "3d_fullres",
    "--preprocessed-root", PREPROCESSED,
    "--results-root", RESULTS,
    "--output-dir", OUTPUT,
    "--bootstrap-repetitions", "10000",
    "--bootstrap-seed", "2026",
]
run_logged("Resumable 149-case NCCT OOF evaluation", command, cwd=REPO)


## 5. Verify completed outputs


In [ ]:
expected = [
    OUTPUT / "isles2024_ncct_oof_case_metrics.csv",
    OUTPUT / "isles2024_ncct_oof_summary.json",
    OUTPUT / "isles2024_ncct_oof_manifest.json",
]
for path in expected:
    assert path.is_file(), f"Missing required result: {path}"
progress = json.loads(PROGRESS.read_text())
summary = json.loads(expected[1].read_text())
assert progress["status"] == "complete", progress
assert progress["completed_cases"] == 149, progress
assert summary["n_oof_cases"] == 149
assert summary["unique_oof_cases"] == 149

print("FINAL STATUS:", progress["status"])
print("PROGRESS:", progress["completed_cases"], "/", progress["total_cases"],
      f"({progress['percent']}%)")
for metric, stats in summary["metrics"].items():
    print(metric, "mean:", round(stats["mean"], 6),
          "95% bootstrap CI:", stats["bootstrap_mean_95ci"])
print("Private results:", OUTPUT)
print("Persistent progress:", PROGRESS)
print("Persistent full log:", LOG)
